In [ ]:
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import FakeEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import json 

In [ ]:
import requests
import os
from tqdm import tqdm

# Download process logs json file from zenodo.
def download_json_from_zenodo(record_id, output_dir='/data'):
    """
    Downloads only .json files from a specific Zenodo record.
    """
    api_url = f"https://zenodo.org/api/records/{record_id}"
   
    try:
        r = requests.get(api_url)
        r.raise_for_status()
        data = r.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching metadata: {e}")
        return

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Scanning record: {data['metadata']['title']}")
   
    files_found = 0
   
    for file_info in data.get('files', []):
        file_name = file_info['key']
       
        # Check if the file ends with .json (case-insensitive)
        if not file_name.lower().endswith('.json'):
            continue
           
        files_found += 1
        download_url = file_info['links']['self']
        file_size = file_info['size']
        output_path = os.path.join(output_dir, file_name)
       
        if os.path.exists(output_path):
            print(f"File {file_name} already exists. Skipping.")
            continue

        print(f"Downloading {file_name}...")
       
        with requests.get(download_url, stream=True) as file_r:
            file_r.raise_for_status()
            with open(output_path, "wb") as f:
                with tqdm(total=file_size, unit='B', unit_scale=True, desc=file_name) as pbar:
                    for chunk in file_r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
                           
    if files_found == 0:
        print("No .json files found in this record.")
    else:
        print(f"Download complete. Processed {files_found} JSON file(s).")


In [ ]:
jq_event_lines = (
    # Find any object with id, type, attributes
    '.. | objects | select(has("id"))'
)

def get_docs(jsonpath):
    loader = JSONLoader(
        file_path=jsonpath,
        jq_schema=jq_event_lines,
        text_content=False
    )

    def jsondic(content):
        try:        
            ev = json.loads(content)  # parse the JSON string    
        except json.JSONDecodeError:        # Not JSON? Then just return as-is.        
            return doc
        return ev

    rag_docs = []
    for doc in loader.lazy_load():
        ev = jsondic(doc.page_content)  # this is the dict: {"id", "type", "attributes", "relationships", ...}

        # Essentials
        ev_id = ev.get("id", "")
        ev_type = ev.get("type", "")

        # Keep only meaningful attribute values (ignore "", None)
        attrs = []
        for a in ev.get("attributes", []):
            v = a.get("value")
            if v not in ("", None):
                # you can also skip the epoch timestamps if you want
                attrs.append(f"{a.get('name') }={v}@{a.get('time')}")

        # Relationships (optional but useful)
        rels = []
        for r in ev.get("relationships", []):
            rels.append(f"{r.get('qualifier')}: {r.get('objectId')}")

        # Build a compact, readable line (ideal for embeddings)
        lines = [f"{ev_id} | {ev_type}"]
        if attrs:
            lines.append("; ".join(attrs))
        if rels:
            lines.append(" | ".join(rels))

        # Replace the dict in page_content with your formatted string
        doc.page_content = " | ".join(lines)

        # Optional: keep minimal, useful metadata (traceability)
        doc.metadata = {
            "id": ev_id,
            "type": ev_type,
        }

        rag_docs.append(doc)

    return rag_docs


In [ ]:
def get_retriever(docs):
    # Chunk loaded documents
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunked_docs = splitter.split_documents(docs)

    # Set embedding model TODO
    embeddings = FakeEmbeddings(size=1352)
    
    # Add documents to db.
    vectorstore = Chroma.from_documents(
        documents=chunked_docs,
        embedding=embeddings,
        persist_directory="./chroma_db" 
    )

    # chromadb retriever
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":5})

    return retriever

In [ ]:
def get_qa_chain(retriever):
    # LLM model
    llm = ChatOpenAI(
        model_name="gpt-4o-mini",
        temperature=0
    )

    # Build RAG chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )

    return qa_chain

In [ ]:
# Usage
RECORD_ID = "8412920"
download_json_from_zenodo(RECORD_ID, output_dir="./zenodo_json_data")

docs = get_docs("zenodo_json_data/ocel2-p2p.json")

retriever = get_retriever(docs)

qa_chain = get_qa_chain(retriever)